In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
import time
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from nnsight import  NNsight
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from datasets import load_dataset

In [2]:
SPLIT = "train"
NUM_EXAMPLES = 100

TASKS_TO_HF_NAMES = {
    'ioi': 'ioi',
    'mcqa': 'copycolors_mcqa',
    'arithmetic_addition': 'arithmetic_addition',
    'arithmetic_subtraction': 'arithmetic_subtraction',
    'arc_easy': 'arc_easy',
    'arc_challenge': 'arc_challenge',
}

In [3]:
list(zip(*[[1, 1], [2, 2]]))

[(1, 2), (1, 2)]

In [4]:
def collate_fn(xs):
    clean, corrupted, labels = zip(*xs)
    clean_labels, corrupt_labes = zip(*labels)
    return list(clean), list(corrupted), clean_labels, corrupt_labes 


class MIBDataset(Dataset):
    """Minimal MIB dataset loader (no transformer_lens dependency)."""

    def __init__(self, task, tokenizer, model_name, split='train', num_examples=None):
        self.task = task
        self.tokenizer = tokenizer
        self.model_name = model_name

        hf_url = f"mib-bench/{TASKS_TO_HF_NAMES[task]}"
        if task == 'mcqa':
            self.dataset = load_dataset(hf_url, '4_answer_choices', split=split)
            self.counterfactual_type = "symbol_counterfactual"
        elif task.startswith('arc'):
            self.dataset = load_dataset(hf_url, split=split)
            self.counterfactual_type = "symbol_counterfactual"
        elif task.startswith('arithmetic'):
            self.dataset = load_dataset(hf_url, split=split)
            self.operator = "-" if "subtraction" in task else "+"
        else:
            self.dataset = load_dataset(hf_url, split=split)

        self.dataset = self._filter()
        if num_examples and num_examples < len(self.dataset):
            self.dataset = self.dataset.select(range(num_examples))

    def _filter(self):
        tok = self.tokenizer
        if self.task == 'ioi':
            return self.dataset.filter(
                lambda x: (
                    len(tok(f" {x['metadata']['indirect_object']}", add_special_tokens=False).input_ids) ==
                    len(tok(f" {x['metadata']['subject']}", add_special_tokens=False).input_ids) and
                    len(tok(f" {x['metadata']['indirect_object']}", add_special_tokens=False).input_ids) ==
                    len(tok(f" {x['metadata']['random_c']}", add_special_tokens=False).input_ids)
                )
            )
        elif self.task == 'mcqa' or self.task.startswith('arc'):
            ct = self.counterfactual_type
            return self.dataset.filter(
                lambda x: (
                    len(tok(x["choices"]["label"][x["answerKey"]], add_special_tokens=False).input_ids) ==
                    len(tok(str(x[ct]["choices"]["label"][x[ct]["answerKey"]]), add_special_tokens=False).input_ids)
                )
            )
        elif self.task.startswith('arithmetic'):
            op = self.operator
            return self.dataset.filter(
                lambda x: (
                    len(tok(str(x["label"]), add_special_tokens=False).input_ids) == 1 and
                    x["random_counterfactual"] is not None and
                    x["random_counterfactual"]["prompt"] is not None and
                    x["operator"] == op and
                    len(tok(str(x["random_counterfactual"]["label"]), add_special_tokens=False).input_ids) == 1
                )
            )
        return self.dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        row = self.dataset[index]
        tok = self.tokenizer

        if self.task == 'ioi':
            correct_idx = tok(f" {row['metadata']['indirect_object']}", add_special_tokens=False).input_ids[0]
            incorrect_idx = tok(f" {row['metadata']['subject']}", add_special_tokens=False).input_ids[0]
            cf = row.get("s2_io_flip_counterfactual", row.get("counterfactual", {}))
            return row["prompt"], cf.get("prompt", row["prompt"]), [correct_idx, incorrect_idx]

        elif self.task == 'mcqa' or self.task.startswith('arc'):
            ct = self.counterfactual_type
            correct_idx = tok(row["choices"]["label"][row["answerKey"]], add_special_tokens=False).input_ids[0]
            cf = row[ct]
            incorrect_idx = tok(str(cf["choices"]["label"][cf["answerKey"]]), add_special_tokens=False).input_ids[0]
            return row["prompt"], cf["prompt"], [correct_idx, incorrect_idx]

        elif self.task.startswith('arithmetic'):
            correct_idx = tok(str(row["label"]), add_special_tokens=False).input_ids[0]
            cf = row["random_counterfactual"]
            incorrect_idx = tok(str(cf["label"]), add_special_tokens=False).input_ids[0]
            return row["prompt"], cf["prompt"], [correct_idx, incorrect_idx]

In [5]:
model_id = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'left'
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, attn_implementation="eager", device_map='auto').eval()
model = NNsight(model)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [6]:
batch_size = 16
dataset = MIBDataset('ioi', tokenizer, model_id, SPLIT)
dataloader = DataLoader(
    dataset, batch_size=batch_size,
    collate_fn=collate_fn, shuffle=False,
)

In [7]:
def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_inverse_rope(grad, rot_embeds):
    """
    Backpropagates gradients through the RoPE operation by applying 
    the rotation in the opposite direction.
    """
    cos, sin = rot_embeds
    cos = cos.unsqueeze(1)
    sin = sin.unsqueeze(1)
    return (grad * cos) + (rotate_half(grad) * (-sin))

def compute_rmsnorm_input_gradients(
    grad: torch.Tensor,
    x_pre_norm: torch.Tensor,
    rmsnorm_weight: torch.Tensor,
    eps: float = 1e-6
):
    D = x_pre_norm.shape[-1]
    if grad.dim() == x_pre_norm.dim() + 1: # accommodate head dimension
        x_pre_norm = x_pre_norm.unsqueeze(1)
    variance = (x_pre_norm ** 2).mean(dim=-1, keepdim=True)
    sigma = torch.sqrt(variance + eps)
    
    u = grad * rmsnorm_weight
    u_dot_x = (u * x_pre_norm).sum(dim=-1, keepdim=True)
    
    term2 = (u_dot_x / (D * (variance + eps))) * x_pre_norm
    grad_pre_norm = (u - term2) / sigma
    
    return grad_pre_norm

def compute_headwise_input_gradients(
    grad_heads: torch.Tensor, 
    W_linear: torch.Tensor, 
    x_pre_norm: torch.Tensor, 
    rmsnorm_weight: torch.Tensor,
    rot_embeds: tuple = None, 
    eps: float = 1e-6
):
    B, H, S, d = grad_heads.shape
    D = x_pre_norm.size(-1)

    if rot_embeds:
        grad_heads = apply_inverse_rope(grad_heads, rot_embeds)
    
    # STEP 1: Backprop through the projection
    W_linear = W_linear.view(-1, d, D)
    if W_linear.size(0) != H: # repeat W_linear for grouped query attention (GQA)
        num_key_value_groups = H // W_linear.size(0)
        W_linear = W_linear.repeat_interleave(num_key_value_groups, dim=0)

    grad_post_norm = torch.einsum('bhsd, hdD -> bhsD', grad_heads, W_linear)
    
    # STEP 2: Backprop through the RMSNorm
    grad_pre_norm = compute_rmsnorm_input_gradients(grad_post_norm, x_pre_norm, rmsnorm_weight, eps)
    
    return grad_pre_norm.transpose(0, 1)


def update_source_2d(source_2d: torch.Tensor, new_tensor: torch.Tensor, type: str, num_heads: int, layer: int = None):
    if type == 'emb':
        source_slice = (0,)
    elif type == 'attn':
        start = 1 + layer * (num_heads + 1)
        source_slice = slice(start, start + num_heads)
    elif type == 'mlp':
        start = 1 + layer * (num_heads + 1) + num_heads
        source_slice = (start,)
    else:
        raise NotImplementedError
    
    source_2d[source_slice] = new_tensor.detach()

def update_scores(scores: torch.Tensor, grad: torch.Tensor, source_2d: torch.Tensor, type: str, num_heads: int, curr_batch_size: int, layer: int = None):
    if type == 'lm_head':
        source_slice = slice(None, None)
    elif type == 'mlp':
        stop = 1 + layer * (num_heads + 1) + num_heads
        source_slice = slice(None, stop)
    elif type.startswith('attn'):
        stop = 1 + layer * (num_heads + 1)
        source_slice = slice(None, stop)
    else:
        raise NotImplementedError(f"Type '{type}' is not implemented.")
    
    if type == 'lm_head':
        grad_slice = (-1,)
    elif type == 'mlp':
        start = layer * (3 * num_heads + 1) + 3 * num_heads
        grad_slice = (start,)
    elif type == 'attn_v':
        start = layer * (3 * num_heads + 1) + 2 * num_heads
        grad_slice = slice(start, start + num_heads)
    elif type == 'attn_k':
        start = layer * (3 * num_heads + 1) + num_heads
        grad_slice = slice(start, start + num_heads)
    elif type == 'attn_q':
        start = layer * (3 * num_heads + 1)
        grad_slice = slice(start, start + num_heads)
    else:
        raise NotImplementedError(f"Type '{type}' is not implemented.")
    
    weight = curr_batch_size / batch_size**2
    new_scores = weight * torch.matmul(source_2d[source_slice], grad.T)
    scores[source_slice, grad_slice] += new_scores.detach()

def run_model(model, batch, scores):
    clean_prompt, _, clean_targets, corrupt_targes = batch
    inputs = tokenizer(clean_prompt, padding=True, return_tensors='pt')

    B, S = inputs['input_ids'].shape
    L = model.config.num_hidden_layers
    H = model.config.num_attention_heads
    d = model.config.head_dim
    D = model.config.hidden_size
    BSD = B * S * D # aggregate dimensions

    source_dims = (1 + L * (H + 1)) # emb + layer * (heads_out + mlp_out)
    grad_dims = (L * (3 * H + 1) + 1) # layer * (heads_in + mlp_in) + lm_head_in


    with model.trace(**inputs) as tracer:
        cache = {i:{} for i in range(-1, model.config.num_hidden_layers)}
        source_2d = torch.zeros(source_dims, BSD, device=model.device, dtype=model.dtype)
        
        cache[-1]['out'] = model.model.embed_tokens.output
        update_source_2d(source_2d=source_2d, new_tensor=cache[-1]['out'].reshape(1, BSD),
            type='emb', num_heads=H)

        rot_cos, rot_sin = model.model.rotary_emb.output
        rot_embeds = (rot_cos.detach(), rot_sin.detach())

        for i, layer in enumerate(model.model.layers):
            layer.self_attn.source # call to build
            cache[i]['q_proj_out'] = layer.self_attn.q_proj.output
            cache[i]['k_proj_out'] = layer.self_attn.source.attention_interface_0.source.repeat_kv_0.output
            cache[i]['v_proj_out'] = layer.self_attn.source.attention_interface_0.source.repeat_kv_1.output

            z = layer.self_attn.source.attention_interface_0.output[0].detach()
            W_linear = layer.self_attn.o_proj.weight.data.T.reshape(H, d, D).detach()
            head_wise_attn_out = torch.einsum('BSHd, HdD -> HBSD', z, W_linear).reshape(H, BSD)
            update_source_2d(source_2d=source_2d, new_tensor=head_wise_attn_out,
                type='attn', num_heads=H, layer=i)

            cache[i]['mid'] = layer.post_attention_layernorm.input

            cache[i]['mlp_in'] = layer.mlp.input
            mlp_out = layer.mlp.output.reshape(1, BSD)
            update_source_2d(source_2d=source_2d, new_tensor=mlp_out,
                type='mlp', num_heads=H, layer=i)

            cache[i]['out'] = layer.output

        clean_logits = model.lm_head.output[range(len(clean_targets)), -1, clean_targets]
        corrupt_logits = model.lm_head.output[range(len(corrupt_targes)), -1, corrupt_targes]

        metric = (clean_logits - corrupt_logits).sum()

        with metric.backward():
            lm_head_grad = cache[i]['out'].grad.detach().reshape(1, BSD)
            update_scores(scores=scores, grad=lm_head_grad, source_2d=source_2d,
                type='lm_head', num_heads=H, curr_batch_size=B)

            for j in range(i, -1, -1):
                layer = model.model.layers[j]
                prev_out = cache[j - 1]['out'].detach()

                mlp_in_grad_pre_norm = compute_rmsnorm_input_gradients(
                    grad=cache[j]['mlp_in'].grad.detach(),
                    x_pre_norm=cache[j]['mid'].detach(),
                    rmsnorm_weight=layer.post_attention_layernorm.weight.data
                ).reshape(1, BSD)
                update_scores(scores=scores, grad=mlp_in_grad_pre_norm, source_2d=source_2d,
                    type='mlp', num_heads=H, curr_batch_size=B, layer=j)
                del mlp_in_grad_pre_norm

                v_proj_in_grad_pre_norm = compute_headwise_input_gradients(
                    grad_heads=cache[j]['v_proj_out'].grad.detach(),
                    W_linear=layer.self_attn.v_proj.weight.data,
                    x_pre_norm=prev_out,
                    rmsnorm_weight=layer.input_layernorm.weight.data,
                ).reshape(H, BSD)
                update_scores(scores=scores, grad=v_proj_in_grad_pre_norm, source_2d=source_2d,
                    type='attn_v', num_heads=H, curr_batch_size=B, layer=j)
                del v_proj_in_grad_pre_norm

                k_proj_in_grad_pre_norm = compute_headwise_input_gradients(
                    grad_heads=cache[j]['k_proj_out'].grad.detach(),
                    W_linear=layer.self_attn.k_proj.weight.data,
                    rot_embeds=rot_embeds,
                    x_pre_norm=prev_out,
                    rmsnorm_weight=layer.input_layernorm.weight.data,
                ).reshape(H, BSD)
                update_scores(scores=scores, grad=k_proj_in_grad_pre_norm, source_2d=source_2d,
                    type='attn_k', num_heads=H, curr_batch_size=B, layer=j)
                del k_proj_in_grad_pre_norm

                q_proj_in_grad_pre_norm = compute_headwise_input_gradients(
                    grad_heads=cache[j]['q_proj_out'].grad.detach().reshape(B, S, H, d).transpose(1, 2),
                    W_linear=layer.self_attn.q_proj.weight.data,
                    rot_embeds=rot_embeds,
                    x_pre_norm=prev_out,
                    rmsnorm_weight=layer.input_layernorm.weight.data,
                ).reshape(H, BSD)
                update_scores(scores=scores, grad=q_proj_in_grad_pre_norm, source_2d=source_2d,
                    type='attn_q', num_heads=H, curr_batch_size=B, layer=j)
                del q_proj_in_grad_pre_norm
        
        del tracer, cache, source_2d

In [ ]:
import gc
L = model.config.num_hidden_layers
H = model.config.num_attention_heads
D = model.config.hidden_size

source_dims = (1 + L * (H + 1)) # emb + layer * (heads_out + mlp_out)
grad_dims = (L * (3 * H + 1) + 1) # layer * (heads_in + mlp_in) + lm_head_in
scores = torch.zeros(source_dims, grad_dims, device=model.device, dtype=model.dtype)

for batch in tqdm(dataloader):
    run_model(model, batch, scores)
    


  0%|          | 0/625 [00:00<?, ?it/s]/tmp/ipykernel_1741654/3721042304.py:109: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  new_scores = weight * torch.matmul(source_2d[source_slice], grad.T)
100%|██████████| 625/625 [05:28<00:00,  1.90it/s]


In [9]:
# embedding (src): input

# attention src: a{layer_id}.h{head_id} e.g. a0.h21
# attention tgt: a{layer_id}.h{head_id}<{head_type}> e.g. a0.h21<k>

# mlp src / tgt: m{layer_id} e.g. m2

# lm_head (tgt): logit

# edge: {src}->{tgt} : {'score': {score}, 'in_graph': False}

# Json Schema
# {
#   "cfg": {
#     "n_layers": 32,
#     "n_heads": 32,
#     "parallel_attn_mlp": False,
#     "d_model": 4096
#   },
#   "nodes": {},
#   "edges": {},
# }

def get_src_id(name: str, n_heads):
    if name == 'input':
        return 0
    if name[0] == 'm':
        layer_id = int(name[1:])
        return 1 + layer_id * (n_heads + 1) + n_heads
    if name[0] == 'a':
        attn_mod, head = name.split('.')
        layer_id = int(attn_mod[1:])
        head_id = int(head[1:])
        return 1 + layer_id * (n_heads + 1) + head_id
    raise NotImplementedError(f"Name '{name}' does not fit the patterns.")

def get_tgt_id(name: str, n_heads):
    if name == 'logits':
        return -1
    if name[0] == 'm':
        layer_id = int(name[1:])
        return layer_id * (3 * n_heads + 1) + 3 * n_heads
    if name[0] == 'a':
        attn_mod, head = name.split('.')
        layer_id = int(attn_mod[1:])
        head_id = int(head[1:-3])
        head_type = head[-2]
        if head_type == 'q':
            return layer_id * (3 * n_heads + 1) + head_id
        if head_type == 'k':
            return layer_id * (3 * n_heads + 1) + n_heads + head_id
        if head_type == 'v':
            return layer_id * (3 * n_heads + 1) + 2 * n_heads + head_id
    raise NotImplementedError(f"Name '{name}' does not fit the patterns.")

def create_mib_circuit(scores: torch.Tensor, n_layers, n_heads, d_model, circuit_level: str = 'edge'):
    circuit = {  
        "cfg": {
            "n_layers": n_layers,
            "n_heads": n_heads,
            "parallel_attn_mlp": False,
            "d_model": d_model
        },
        'nodes': {},
        'edges': {}
    }


    def _inner_loop(src_name: str, src_layer_id: int = 0, src_head_id: int = None):
        src_id = get_src_id(src_name, n_heads)
        for tgt_layer_id in range(src_layer_id, n_layers):

            if tgt_layer_id > src_layer_id or src_name == 'input':
                for tgt_head_type in ['q', 'k', 'v']:
                    for tgt_head_id in range(n_heads):
                        tgt_name = f"a{tgt_layer_id}.h{tgt_head_id}<{tgt_head_type}>"
                        tgt_id = get_tgt_id(tgt_name, n_heads)
                        circuit['edges'][f"{src_name}->{tgt_name}"] = {'score': scores[src_id, tgt_id].item(), 'in_graph': False}

            if tgt_layer_id > src_layer_id and not src_name.startswith('m'):
                tgt_name = f"m{tgt_layer_id}"
                tgt_id = get_tgt_id(tgt_name, n_heads)
                circuit['edges'][f"{src_name}->{tgt_name}"] = {'score': scores[src_id, tgt_id].item(), 'in_graph': False}
        
        circuit['edges'][f"{src_name}->logits"] = {'score': scores[src_id, -1].item(), 'in_graph': False}

    
    # outer loop
    _inner_loop("input") 
    circuit["nodes"]["input"] = {'in_graph': False}

    for layer_id in range(n_layers):
        for head_id in range(n_heads):
            src_name = f"a{layer_id}.h{head_id}"
            _inner_loop(src_name, src_layer_id=layer_id, src_head_id=head_id)
            circuit["nodes"][src_name] = {'in_graph': False}
        src_name = f"m{layer_id}"
        _inner_loop(src_name, src_layer_id=layer_id)
        circuit["nodes"][src_name] = {'in_graph': False}


    return circuit

In [12]:
circuit = create_mib_circuit(scores, L, H, D)
import json
with open('test_importance.json', 'w') as f:
    json.dump(circuit, f, indent=2)

In [ ]:
"""
CUDA_VISIBLE_DEVICES=1 python experiments/mib/MIB-circuit-track/run_evaluation.py --models llama3 --tasks ioi --split test --batch-size 8 --circuit-files /home/dacslab/lasse_jantsch/circuit_discovery/circuits/test_importance.json --output-dir /home/dacslab/lasse_jantsch/circuit_discovery/experiments/mib/results

python print_results.py --output-dir /home/dacslab/lasse_jantsch/circuit_discovery/experiments/mib/MIB-circuit-track/results --split test --metric cpr
"""